# 🦾 Titan Browser — GitHub Actions Runner on Colab

Ce notebook transforme ta session Colab en **self-hosted runner** pour GitHub Actions.  
Le build Chromium tourne ici (jusqu'à 12h session gratuite) au lieu du runner GitHub (limité à 6h).

### Étapes
1. **Cellule 1** — Colle ton `GITHUB_PAT` (token avec `repo` + `workflow` scopes)
2. **Cellule 2** — Vérifie les ressources disponibles
3. **Cellule 3** — Installe et lance le runner (tourne en arrière-plan)
4. Dans GitHub → repo → Actions → **Run workflow** → sélectionne **`self-hosted`**

> ⚠️ **Ne ferme pas l'onglet Colab** pendant le build — la session s'arrête si le navigateur est fermé trop longtemps.
> Active **"Keep session alive"** dans les paramètres Colab (Tools → Settings → Other).

In [ ]:
# ── Cellule 1 : Configuration ─────────────────────────────────────────────────
import os

# Colle ton Personal Access Token GitHub ici
# Scopes nécessaires : repo, workflow, admin:org (ou juste repo+workflow pour un repo perso)
GITHUB_PAT = ""  # <-- ton token ici

REPO = "ferelking242/titan-browser"
RUNNER_NAME = "colab-runner"
RUNNER_LABELS = "self-hosted,colab,linux,x64"

assert GITHUB_PAT, "❌ Colle ton GITHUB_PAT ci-dessus !"
os.environ['GITHUB_PAT'] = GITHUB_PAT
print("✅ Config OK")

In [ ]:
# ── Cellule 2 : Vérification des ressources ───────────────────────────────────
import subprocess, shutil

def sh(cmd):
    return subprocess.check_output(cmd, shell=True, text=True).strip()

print("=== CPU ===")
print(sh("nproc && lscpu | grep 'Model name' | head -1"))

print("\n=== RAM ===")
print(sh("free -h | head -2"))

print("\n=== Disk ===")
print(sh("df -h / | tail -1"))

# Chromium build needs ~80GB total
_, _, free = shutil.disk_usage("/")
free_gb = free / (1024**3)
if free_gb < 60:
    print(f"\n⚠️  Seulement {free_gb:.0f}GB libres — Chromium en a besoin de ~80GB.")
    print("   Dans Colab : Runtime → Disconnect and delete runtime, puis reconnecte.")
else:
    print(f"\n✅ {free_gb:.0f}GB libres — suffisant pour le build")

In [ ]:
# ── Cellule 3 : Installation du runner GitHub Actions ─────────────────────────
import requests, subprocess, os, tarfile, urllib.request

RUNNER_VERSION = "2.325.0"
RUNNER_DIR = "/root/actions-runner"

# 1. Obtenir un token d'enregistrement via l'API GitHub
print("Récupération du token d'enregistrement...")
resp = requests.post(
    f"https://api.github.com/repos/{REPO}/actions/runners/registration-token",
    headers={
        "Authorization": f"token {GITHUB_PAT}",
        "Accept": "application/vnd.github.v3+json"
    }
)
assert resp.status_code == 201, f"❌ Erreur API: {resp.status_code} — {resp.text}"
reg_token = resp.json()["token"]
expires = resp.json()["expires_at"]
print(f"✅ Token obtenu (expire: {expires})")

# 2. Télécharger le runner
os.makedirs(RUNNER_DIR, exist_ok=True)
runner_url = f"https://github.com/actions/runner/releases/download/v{RUNNER_VERSION}/actions-runner-linux-x64-{RUNNER_VERSION}.tar.gz"
runner_tar = f"/tmp/actions-runner.tar.gz"

print(f"\nTéléchargement du runner v{RUNNER_VERSION}...")
urllib.request.urlretrieve(runner_url, runner_tar)
print("✅ Téléchargé")

# 3. Extraire
print("Extraction...")
subprocess.run(["tar", "xzf", runner_tar, "-C", RUNNER_DIR], check=True)
print("✅ Extrait")

# 4. Installer les dépendances système du runner
print("\nInstallation des dépendances runner...")
subprocess.run(
    f"cd {RUNNER_DIR} && bash bin/installdependencies.sh",
    shell=True, check=True, capture_output=True
)
print("✅ Dépendances OK")

In [ ]:
# ── Cellule 4 : Enregistrement + démarrage du runner ─────────────────────────
import subprocess, threading, time

RUNNER_DIR = "/root/actions-runner"

# Enregistrement
print("Enregistrement du runner...")
reg_cmd = (
    f"cd {RUNNER_DIR} && "
    f"./config.sh "
    f"--url https://github.com/{REPO} "
    f"--token {reg_token} "
    f"--name {RUNNER_NAME} "
    f"--labels '{RUNNER_LABELS}' "
    f"--work /root/_work "
    f"--replace "
    f"--unattended"
)
result = subprocess.run(reg_cmd, shell=True, capture_output=True, text=True)
if result.returncode != 0:
    print("❌ Erreur:", result.stderr)
    raise RuntimeError("Enregistrement échoué")
print("✅ Runner enregistré")

# Démarrage en arrière-plan avec logs
print("\nDémarrage du runner (Ctrl+C pour arrêter)...")
print("─" * 60)
print("👉 Va sur GitHub → ton repo → Actions → Run workflow")
print("   Sélectionne runner: self-hosted (ou colab)")
print("─" * 60)

# Lance le runner (bloquant — logs visibles dans la cellule)
proc = subprocess.Popen(
    f"cd {RUNNER_DIR} && ./run.sh",
    shell=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

try:
    for line in proc.stdout:
        print(line, end="")
except KeyboardInterrupt:
    proc.terminate()
    print("\n🛑 Runner arrêté")

---
## ℹ️ Notes importantes

### Éviter la déconnexion Colab
Colab déconnecte les sessions inactives. Pendant le build (3-6h), le runner est actif donc Colab ne devrait pas se déconnecter. Mais pour être sûr :
- **Ne ferme pas le navigateur**
- **Tools → Settings → Other → coche "Auto connect to runtime"**

### Ressources Colab
| Tier | CPU | RAM | Session max |
|------|-----|-----|-------------|
| Gratuit | 2 vCPU | 12 GB | ~12h |
| Pro | 2-4 vCPU | 25 GB | 24h |
| Pro+ | 4-8 vCPU | 52 GB | 24h |

Avec 2 vCPU (gratuit), le build arm64 prend ~8-10h → session Pro recommandée pour passer sous 6h.  
Avec ccache chaud (2ème build), n'importe quel tier suffit (~1-2h).

### Après le build
L'APK est publié automatiquement dans **Releases** du repo GitHub.  
Le runner peut être supprimé depuis **Settings → Actions → Runners**.

### Déclencher le build
```
GitHub → ferelking242/titan-browser → Actions → Build → Run workflow
Runner: self-hosted
Full build: ☐ (arm64 only, recommandé)
```